# AlphaLens — Lesson 1
## Does inflation change which types of stocks perform best?

Welcome! **You do not need to know Python yet.** Read each note, then click the ▶ beside each code cell from top to bottom. Today we will load **real data**, compare six kinds of stocks, and make our first chart. Nothing here is investment advice.

## Step 1 — Import Python tools
`pandas` works with tables. `matplotlib` draws charts. `display` prints a friendly table. Colab usually has these libraries preinstalled.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print("✅ Python is working! Your first AlphaLens notebook is ready.")

## Step 2 — Download historical stock portfolio returns
We use six **research portfolios** from professor Kenneth French's free Dartmouth data library. These are groups of U.S. stocks, sorted by company size (small/big) and book-to-market ratio (a rough value/growth indicator). We are **not** picking six individual stocks. The returns are monthly percentages.

In [ ]:
"""AlphaLens data loader. Returns percent monthly portfolio returns and CPI levels."""
import io
import re
import urllib.request
import zipfile
import pandas as pd

FRENCH_URL = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/6_Portfolios_2x3_CSV.zip'
FRED_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL'
PORTFOLIOS = ['Small Growth', 'Small Neutral', 'Small Value', 'Big Growth', 'Big Neutral', 'Big Value']



def fetch_french_monthly(url=FRENCH_URL):
    """Download Kenneth French six size/value portfolios, percent monthly returns."""
    req = urllib.request.Request(url, headers={'User-Agent': 'AlphaLens educational research (public data)'})
    with urllib.request.urlopen(req, timeout=40) as response:
        payload = response.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        csv_name = next((n for n in archive.namelist() if n.lower().endswith('.csv')), None)
        if csv_name is None:
            raise ValueError('Kenneth French archive did not contain a CSV file')
        content = archive.read(csv_name).decode('utf-8-sig', errors='replace')
    # File starts with notes; the first block of YYYYMM, six monthly returns is value-weighted monthly portfolios.
    rows = []
    for line in content.splitlines():
        parts = [p.strip() for p in line.split(',')]
        if len(parts) >= 7 and re.fullmatch(r'\d{6}', parts[0]):
            if rows and int(parts[0]) <= int(rows[-1][0]):
                break  # next section: do not accidentally join equal-weight or annual data
            rows.append(parts[:7])
        elif rows:
            break
    if len(rows) < 120:
        raise ValueError(f'Unexpected French CSV format: only {len(rows)} monthly rows')
    df = pd.DataFrame(rows, columns=['month'] + PORTFOLIOS)
    df.index = pd.PeriodIndex(df.pop('month'), freq='M')
    for column in PORTFOLIOS:
        df[column] = pd.to_numeric(df[column], errors='coerce')
        df.loc[df[column] <= -99, column] = float('nan')  # missing-value codes
    df.index.name = 'month'
    return df


returns = fetch_french_monthly()
print(f"✅ Loaded {len(returns)} months of stock portfolio data.")
display(returns.tail())

## Step 3 — Download U.S. inflation data
The Consumer Price Index (CPI) measures changes in prices paid by consumers. We turn the CPI index into **year-over-year inflation**: how much prices changed compared with the same month a year earlier.

In [ ]:
FRED_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL"
raw = pd.read_csv(FRED_URL)
date_column = "DATE" if "DATE" in raw.columns else "observation_date"
if date_column not in raw.columns or "CPIAUCSL" not in raw.columns:
    raise ValueError(f"Unexpected FRED columns: {list(raw.columns)}")
cpi = pd.DataFrame({"CPI": pd.to_numeric(raw["CPIAUCSL"], errors="coerce").to_numpy()},
                   index=pd.PeriodIndex(pd.to_datetime(raw[date_column]), freq="M"))
cpi.index.name = "month"
cpi["inflation_yoy"] = cpi["CPI"].pct_change(12, fill_method=None) * 100
print("✅ Inflation data loaded")
display(cpi.tail())

## Step 4 — Match dates fairly
A CPI number for a given month is published later. We therefore classify a stock-return month using CPI inflation from **two observation months earlier**. This is an easy-to-use approximation, *not* a historically exact record of published figures. We call inflation **high** when it exceeds 3% and **low** otherwise. The 3% cutoff is our choice, not an official classification.

In [ ]:
cpi["known_inflation"] = cpi["inflation_yoy"].shift(2)
data = returns.join(cpi[["known_inflation"]], how="inner")
data = data.loc["2000-01":].dropna()
data["regime"] = data["known_inflation"].apply(lambda x: "High inflation" if x > 3 else "Low inflation")
print(f"✅ {len(data)} months matched: {data.index.min()} through {data.index.max()}")
print(data["regime"].value_counts())
display(data.tail())

## Step 5 — What happened historically?
We calculate each portfolio's **average monthly return** in high and low inflation periods. An average monthly return of 1% does **not** mean the portfolio earned exactly 12% in a year. For now this is a **description**, not a significance test or forecast.

In [ ]:
means = data.groupby("regime")[PORTFOLIOS].mean().T
if not {"High inflation", "Low inflation"}.issubset(means.columns):
    raise ValueError("Both regimes are needed. Try using a longer sample period.")
print("Average MONTHLY stock portfolio return (%)")
display(means.round(3))
ax = means.plot(kind="bar", figsize=(11,5), rot=30)
ax.set_ylabel("Average monthly return (%)")
ax.set_title("Historical returns by stock type and inflation regime")
ax.axhline(0, color="black", linewidth=0.7)
plt.tight_layout()
plt.show()

## Step 6 — Your first research observation
Which group had the highest average return during high inflation? Did that ranking change during low inflation? **Remember:** these are historical differences; we have not tested if they are reliable rather than chance. Do not say inflation *caused* the results.

In [ ]:
high_winner = means["High inflation"].idxmax()
low_winner = means["Low inflation"].idxmax()
print("Highest historical average monthly return in high inflation:", high_winner)
print("Highest historical average monthly return in low inflation:", low_winner)
print()
print("WARNING: Historical conditional averages are not predictions or evidence of causation.")

## What to do after running this notebook
1. Save your own copy using **File → Save a copy in Drive**.
2. Note the two named stock groups and whether the rankings changed.
3. Share the outputs or any error with me; we will inspect the figures together.
4. In Lesson 2 we will add tests of differences, alternative inflation cutoffs, and the Fama–French model.

**Data links:** [Kenneth French](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) · [FRED CPI](https://fred.stlouisfed.org/series/CPIAUCSL).